# Milestone 4 - System Prototype

**Project:** NLP-assisted job opportunity matching for MSBA international students

**Team GitHub notebook URL:** https://github.com/Kongbai815/job-matching-nlp/blob/main/MSBA_Job_Matching_Milestone4_System_Prototype.ipynb

This completed notebook runs a real input-to-output retrieval and grounded-generation pipeline over the team's 100,000-posting project sample.

## 1. Architecture and Data

The prototype follows **input -> retrieval -> decision -> grounded output -> evaluation**. A sparse index retrieves comparable postings from 80,000 training records. A transparent role-fit rubric predicts one of four advisor triage labels. The output cites retrieved title, company, location, skills, and label-reason fields, while work-authorization questions are always escalated when the source lacks evidence.

In [1]:
from pathlib import Path
import json
import pandas as pd
from msba_job_matcher.core import JobMatchingSystem

DATA_PATH = Path('data_jobs_msba_project_sample_100k.csv')
RESULTS_PATH = Path('outputs/milestone4_prototype_results.json')
CASEBOOK_PATH = Path('outputs/final_casebook_outputs.json')
df = pd.read_csv(DATA_PATH)
results = json.loads(RESULTS_PATH.read_text())
casebook_payload = json.loads(CASEBOOK_PATH.read_text())
print('Loaded project rows:', len(df))
print('Training retrieval corpus:', (df['split'] == 'train').sum())
print('Held-out validation rows:', (df['split'] == 'validation').sum())
print('Prototype evaluation rows:', results['prototype_evaluation']['validation_rows'])


Loaded project rows: 100000
Training retrieval corpus: 80000
Held-out validation rows: 20000
Prototype evaluation rows: 4000


## 2. Four Real Workflow Inputs

The cases cover the primary advisor workflow and three expected failure or escalation paths: a strong entry-level analytics search, a senior engineering mismatch, missing authorization evidence, and thin metadata.

In [2]:
system = JobMatchingSystem(data_path=str(DATA_PATH), top_k=6)
cases = [
    {'case_id': 'entry_level_analytics', 'query': 'Find entry-level US data analyst, business analyst, or BI roles for an MSBA student with SQL, Python, Excel, Tableau, and Power BI.'},
    {'case_id': 'senior_engineering_filter', 'query': 'Evaluate a senior data engineer role requiring seven years, Spark, AWS, Kubernetes, and Scala for an entry-level MSBA student.'},
    {'case_id': 'authorization_missing', 'query': 'Does this US data analyst posting definitely support CPT, OPT, or H-1B when no authorization text is present?'},
    {'case_id': 'thin_metadata', 'query': 'Analytics internship with Excel and dashboards, but company, location, detailed skills, and authorization are missing.'},
]
case_results = [{'case_id': c['case_id'], **system.run(c['query'])} for c in cases]
for item in case_results:
    labels = [e['grounded_fit_label'] for e in item['retrieved_evidence'][:3]]
    print(f"{item['case_id']}: query={item['predicted_query_label']}; retrieved={len(item['retrieved_evidence'])}; top evidence labels={', '.join(labels)}")


entry_level_analytics: query=high_fit; retrieved=5; top evidence labels=high_fit, high_fit, high_fit
senior_engineering_filter: query=low_fit; retrieved=5; top evidence labels=medium_fit, medium_fit, unclear
authorization_missing: query=unclear; retrieved=5; top evidence labels=high_fit, high_fit, high_fit
thin_metadata: query=unclear; retrieved=5; top evidence labels=high_fit, medium_fit, medium_fit


## 3. Retrieved Evidence and Grounded Output

In [3]:
for case in casebook_payload['cases']:
    print(f"\nCASE: {case['case_id']} | predicted query label: {case['predicted_query_label']}")
    for item in case['retrieved_evidence'][:3]:
        print(f"  #{item['rank']} {item['role_title']} | {item['company']} | {item['location']} | {item['grounded_fit_label']} | score={item['score']:.3f} | authorization={item['authorization_evidence']}")
    print('  guardrail:', case['grounding_caveat'])



CASE: entry_level_analytics | predicted query label: high_fit
  #1 Data Analyst | HireMatch | Anywhere | high_fit | score=8.054 | authorization=not available in public source
  #2 Data Analyst II, Bay Area - Remote | Astreya | Anywhere | high_fit | score=7.951 | authorization=not available in public source
  #3 Data Analyst (US REMOTE) | LeanTaaS | Anywhere | high_fit | score=6.858 | authorization=not available in public source
  guardrail: Do not infer CPT, OPT, or sponsorship from this public dataset. Authorization evidence must be reviewed separately.

CASE: senior_engineering_filter | predicted query label: low_fit
  #3 Adidas Recruitment 2023 - 2+Years Experience Required -  Analyst Post | Adidas | Anywhere | medium_fit | score=6.482 | authorization=not available in public source
  #6 Technical Data Analyst (6+ Years Experience) | HARRISS CONSULTANCY AND ENTERPRISE SOLUTIONS | Hyderabad, Telangana, India | medium_fit | score=6.366 | authorization=not available in public source
  

## 4. Preliminary Evaluation Against Baseline

The fair preliminary comparison uses the same balanced 4,000-row validation subset. The prototype is also reported beside the Milestone 2 full-validation result, but that cross-set comparison is descriptive rather than causal.

In [4]:
baseline = results['baseline_reference']
metrics = results['prototype_evaluation']['metrics']
print('Same-eval hashed baseline accuracy:', round(baseline['same_eval_hashed_centroid_accuracy'], 4))
print('Same-eval hashed baseline macro F1:', round(baseline['same_eval_hashed_centroid_macro_f1'], 4))
print('M4 prototype accuracy:', round(metrics['accuracy'], 4))
print('M4 prototype macro F1:', round(metrics['macro_f1'], 4))
print('Macro F1 improvement:', round(metrics['macro_f1'] - baseline['same_eval_hashed_centroid_macro_f1'], 4))


Same-eval hashed baseline accuracy: 0.7965
Same-eval hashed baseline macro F1: 0.7914
M4 prototype accuracy: 0.8005
M4 prototype macro F1: 0.8009
Macro F1 improvement: +0.0095


### Per-label performance and error concentration

In [5]:
for label, values in metrics['per_label'].items():
    print(f"{label:10s} precision={values['precision']:.3f} recall={values['recall']:.3f} f1={values['f1']:.3f}")
print('Total errors:', results['prototype_evaluation']['error_analysis']['total_errors'])


high_fit   precision=0.930 recall=0.993 f1=0.960
medium_fit precision=0.892 recall=0.612 f1=0.726
low_fit    precision=0.596 recall=0.937 f1=0.729
unclear    precision=0.979 recall=0.660 f1=0.789
Total errors: 798


The strongest class is `high_fit` (F1 0.960). The main weakness is boundary handling: 380 `medium_fit` rows are pushed to `low_fit`, and 254 `unclear` rows are pushed to `low_fit`. These errors matter because an overly conservative system may hide viable opportunities. The first deployment threshold should therefore prioritize high-fit precision and route borderline cases to review rather than automatically reject them.

## 5. Grounding, Hallucination, and Governance Checks

In [6]:
checks = casebook_payload['governance_checks']
print('Cases evaluated:', checks['cases_evaluated'])
print('Cases with evidence:', checks['cases_with_retrieved_evidence'])
print('Retrieved records checked:', checks['retrieved_records_checked'])
print('Explicit authorization evidence records:', checks['explicit_authorization_evidence_records'])
print('Unsupported authorization claims:', checks['unsupported_authorization_claims'])
print('Authorization caveat rate:', f"{checks['authorization_caveat_rate']:.1%}")


Cases evaluated: 4
Cases with evidence: 4
Retrieved records checked: 20
Explicit authorization evidence records: 1
Unsupported authorization claims: 0
Authorization caveat rate: 100.0%


The top governance risk is an unsupported work-authorization claim. The system surfaces explicit authorization phrases when they appear in retrieved source text and otherwise uses a missing-evidence caveat; it never converts absence of evidence into eligibility. The second risk is automation harm from weak-label overconfidence. The system therefore shows retrieved evidence, distinguishes role fit from authorization, and leaves forwarding or rejection to the advisor. This grounded pattern follows the RAG principle in HOLLM Chapter 8 and the QA/extraction emphasis in Tunstall Chapter 7 and Jurafsky & Martin Chapter 11: retrieval should constrain the answer and make its support inspectable.